# Notebook 02: candidate dsRNA regions against Dmai dsRNase-2

Rational selection of candidate regions for a **nominal 400 nt dsRNA** against Dmai dsRNase-2, with a
**published experimental benchmark of 330 nt** and a length-sensitivity analysis at 300 and 500 nt. The
nominal length is a project convention, not a biological optimum; the plausible space is 300-500 nt.

This notebook is read-only: it loads and displays the stored results of checkpoints 1 to 5 and verifies
the NB01 -> NB02 contract. The scientific logic lives in `zeaguard.nb02_*`; the criteria are frozen in
`config/nb02_design_criteria.yaml` and were committed before any candidate window existed (ADR 0005).

NB02 selects **region and length**. Molecular architecture (linear, hairpin, protected-end, dual-loop),
primers, promoter, loop, terminator, restriction sites, codon optimisation, plasmid backbone, ecological
off-target and any efficacy prediction are **outside** NB02.

In [ ]:
import zeaguard.repro as repro

repro.assert_environment_ready()

## Stage 1: the NB01 -> NB02 contract

`data/reference/nb01_dsrnase2_operational_reference.json` is the versioned hand-off from NB01. Nothing in
it is trusted blindly: the CDS of both dsRNase-2 records is re-extracted from the hash-validated TSA, the
hashes are recomputed, the seven known variable positions are re-derived and the published benchmark
primers are re-found, with a hard failure on any divergence. `GITV01008430.1` is the operational
reference because it reproduces the published CDS and protein exactly; this is not a claim of unique
biological identity, and `GITV01012450.1` is not treated as incorrect (NB01 remains `AMBIGUOUS`).

In [ ]:
from zeaguard import nb02_criteria, nb02_reference
from zeaguard.nb01_dsrnase_investigation import format_table

project_root = repro.find_project_root()
contract = nb02_reference.verify_reference(project_root)
print("operational:", contract["operational"])
print("sibling:    ", contract["sibling"])
print("pin sha256: ", contract["pin_sha256_lf"])
print("NB01 hand-off cross-check:", contract["nb01_handoff_crosscheck"]["status"])

In [ ]:
columns = ["cds_pos", "codon_index", "codon_position", "operational_base", "sibling_symbol", "class",
           "operational_amino_acid", "sibling_possible_amino_acids"]
print(format_table(contract["variants"], columns))

In [ ]:
benchmark = contract["benchmark"]
print("published benchmark:", benchmark["cds_start"], "-", benchmark["cds_end"],
      f"({benchmark['length_nt']} nt)", benchmark["sequence_sha256"][:12])
print("known variants inside:", benchmark["variants_inside"])
print("primer sites in the sibling record:", benchmark["sibling_primer_sites"])
print("published cDNA body:", benchmark["published_cdna_body"])

## Stage 2: the candidate space and its descriptors

Every interval of 300-500 nt inside the 1425 nt operational CDS (step 1 nt, no padding) is a candidate
window: 206,226 in total. Four length strata are described in full: 400 nt (ranking), 330 nt (benchmark
comparator) and 300 and 500 nt (sensitivity). Composition, known variants, potential small-RNA windows
and position are descriptors; the 5'/central/3' label never orders or groups anything.

In [ ]:
import json

results = project_root / "results/bioinformatics/nb02"
cp2 = json.loads((results / "run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(cp2["counts"], indent=1))
print("strata:", cp2["parameters"]["strata"])
print("not done at this stage:", cp2["not_done"])

In [ ]:
from zeaguard.nb01_dsrnase_investigation import read_tsv

summary = read_tsv(results / "window_space_summary.tsv")
focus = [row for row in summary if int(row["length_nt"]) in (300, 330, 400, 500)]
print(format_table(focus, ["length_nt", "n_windows", "gc_min", "gc_max", "n_variant_free_windows",
                           "min_variant_sites", "max_variant_sites", "min_fraction_unaffected"]))

## Stage 3: specificity evidence

The operational CDS is searched against one combined database (the whole *D. maidis* TSA plus the
published cDNAs) with the pre-registered BLASTn settings. Every alignment is rebuilt from BTOP and each
candidate is evaluated **only on the part of the alignment inside its own interval**, so a region is
never penalised for homology outside the sequence that would be produced as dsRNA.

Biological units were frozen before the first search: DSRNASE1 and DSRNASE3 (PARALOG) and BICC
(CO_TARGET). Evidence is aggregated within a unit and never pooled across units. OTHER_TRANSCRIPT is a
descriptor with a **mandatory pre-recommendation manual review**. E-values and exact 19-mer and 21-mer
counts are descriptors, never cutoffs; the five shuffled controls are a reproducible sanity check only.

In [ ]:
cp3 = json.loads((results / "cp3/run_manifest.json").read_text(encoding="utf-8"))
print("BLAST:", cp3["blast"]["version"])
print("arguments:", " ".join(cp3["blast"]["arguments"]))
print("database:", cp3["database"]["n_sequences"], "sequences,", cp3["database"]["total_bases"], "bases")
print(json.dumps(cp3["hsps_by_class_operational_query"], indent=1))

In [ ]:
controls = read_tsv(results / "cp3/shuffled_control_summary.tsv")
print(format_table(controls, ["query", "n_hsps_total", "n_hsps_with_exact_21mer", "longest_exact_match_full_max",
                              "n_hsps_OTHER_TRANSCRIPT", "DSRNASE1_windows_with_any_hsp",
                              "DSRNASE3_windows_with_any_hsp", "BICC_windows_with_any_hsp"]))
print("\nk-mer self-check (BLAST/BTOP versus a direct exact scan):")
print(json.dumps({k: v for k, v in cp3["kmer_selfcheck"].items() if v["blast_btop"] or v["direct_scan"]}, indent=1))

## Stage 4: contiguous equivalence cells and the provisional candidates

A cell is a **maximal contiguous run** of adjacent windows of one stratum sharing the same decisive
signature (intercepted variants plus the clipped DSRNASE1, DSRNASE3 and BICC vectors). With a step of
1 nt, two consecutive windows join the same cell only when `next.cds_start == current.cds_start + 1` and
the signatures are equal; a signature that reappears after an interruption starts a new cell, so
spatially disconnected regions are never merged.

Cells are ordered by the pre-registered hierarchy: structural requirements, then specificity by Pareto
dominance within one stratum, then the intercepted variant count. Descriptors never order, there is no
composite score and no weights, and ties are preserved.

In [ ]:
cp4 = json.loads((results / "cp4/run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(cp4["counts"], indent=1))
print("cell rule:", cp4["cell_rule"])
print("cell signature:", cp4["cell_signature"])

In [ ]:
cell_rows = read_tsv(results / f"cp4/cells_L{cp4['ranking_stratum_nt']}.tsv")
print(format_table(sorted(cell_rows, key=lambda r: int(r["ordering_group"])),
                   ["cell_id", "signature_id", "ordering_group", "n_windows", "start_min", "start_max",
                    "end_min", "end_max", "n_variant_sites", "variant_positions", "in_pareto_front"]))

In [ ]:
review = read_tsv(results / "cp4/other_transcript_review_evidence.tsv")
print(f"{len(review)} OTHER_TRANSCRIPT HSPs clipped to the provisional candidates "
      "(inspection order only, not a decision rule):")
print(format_table(review[:12], ["candidate_id", "subject", "candidate_qstart", "candidate_qend",
                                 "aligned_nt_clipped", "longest_exact_match_clipped", "local_identity_clipped"]))

## Stage 5: the versioned Wet Lab hand-off

A candidate becomes `RECOMMENDED_SHORTLIST` only when a versioned `APPROVED` OTHER_TRANSCRIPT review
decision exists for it in `config/nb02_manual_review_decisions.tsv`; a strong intraspecies concern
excludes it after review and selection moves to the next eligible candidate. The published 330 nt
benchmark stays in a separate `REFERENCE_SET`: it appears in every table, is evaluated with the same
criteria and never counts as a recommended candidate.

In [ ]:
cp5 = json.loads((results / "cp5/run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(cp5["lists"], indent=1))
print(json.dumps(cp5["review"], indent=1))
print(json.dumps(cp5["convergence_with_benchmark"], indent=1))

In [ ]:
from zeaguard import nb02_handoff

handoff = read_tsv(project_root / nb02_handoff.HANDOFF_TSV)
print(format_table(handoff, ["candidate_id", "list_membership", "cds_start", "cds_end", "length_nt",
                             "n_variant_sites", "target_region_position", "gc_fraction",
                             "other_transcript_longest_exact_match_clipped",
                             "other_transcript_exact_21mer_count_clipped", "overlap_with_benchmark_nt"]))

In [ ]:
for row in handoff:
    print(f"== {row['candidate_id']} ({row['list_membership']})")
    print("   rationale:  ", row["inclusion_rationale"])
    print("   limitations:", row["limitations"])
    print()

Stage 5 ends with the hand-off versioned at `data/reference/nb02_dsrnase2_candidate_regions.tsv` and
`.fasta` (coding/sense orientation, no tails). The convergence of `D2-L400-0817-1216` with the published
benchmark (327 nt of overlap) is recorded but **did not participate in the ranking**: the benchmark is
not part of the ordering and never counts as a recommended candidate.

What this notebook does **not** establish: biological efficacy of any region, an optimal dsRNA length,
that 5', central or 3' regions differ, that absence of detected homology is safety, or the identity of
the dsRNase-2 locus (NB01 remains `AMBIGUOUS` at accession, CDS and protein level). The experimental
protocols of the benchmark are `PROJECT_PROVIDED_NOT_AGENT_VERIFIED`. Ecological off-target against
non-target organisms is a later stage. See the run record and ADR 0005 for the full limitations.